In [0]:
# Load raw data from Unity Catalog
df = spark.table("indian_roads.raw.accidents")

# Convert to Pandas — we'll do all cleaning in Pandas
pdf = df.toPandas()

print(f"Shape: {pdf.shape}")
print(f"\nData types:\n{pdf.dtypes}")
print(f"\nMissing values:\n{pdf.isnull().sum()}")

In [0]:
# Check what unique values festival actually contains
print("Unique values in festival column:")
print(pdf['festival'].unique())
# .unique() returns every distinct value in the column
# This will reveal if NaN is stored as a string "nan" 
# rather than a real null value

print(f"\nValue counts:")
print(pdf['festival'].value_counts(dropna=False))
# dropna=False → makes sure NaN values are counted too
# without this, pandas silently ignores NaN in value_counts

# Convert date from string to proper datetime
pdf['date'] = pd.to_datetime(pdf['date'])
# pd.to_datetime() → tells pandas "treat this column as a real date"
# Before: "2023-10-22" was just text
# After:  it becomes a proper date object we can do maths on

# Extract useful date features
pdf['month']      = pdf['date'].dt.month
pdf['year']       = pdf['date'].dt.year
pdf['day']        = pdf['date'].dt.day

# .dt is the "datetime accessor" — lets you pull parts out of a date
# .dt.month → extracts just the month number (1–12)
# .dt.year  → extracts just the year
# .dt.day   → extracts just the day of the month

print("Date column fixed:")
print(pdf[['date', 'year', 'month', 'day']].head(5))

In [0]:
# Check what unique values festival actually contains
print("Unique values in festival column:")
print(pdf['festival'].unique())
# .unique() returns every distinct value in the column
# This will reveal if NaN is stored as a string "nan" 
# rather than a real null value

print(f"\nValue counts:")
print(pdf['festival'].value_counts(dropna=False))
# dropna=False → makes sure NaN values are counted too
# without this, pandas silently ignores NaN in value_counts

In [0]:
# Convert festival to a binary flag — 1 = festival day, 0 = normal day
pdf_clean['is_festival'] = (pdf_clean['festival'] != 'None').astype(int)
# pdf_clean['festival'] != 'None' 
#   → checks every row: is the value something other than "None"?
#   → returns True for Holi/Diwali/Eid/New Year, False for "None"
# .astype(int) 
#   → converts True → 1, False → 0

# Drop the original festival column — we don't need it anymore
pdf_clean = pdf_clean.drop(columns=['festival'])
# we already have is_festival which captures all the information
# in a cleaner, model-friendly format

# Verify
print("Festival flag distribution:")
print(pdf_clean['is_festival'].value_counts())
print(f"\nColumns now: {list(pdf_clean.columns)}")

In [0]:
# Define the mapping — order matters here
# We assign numbers that reflect increasing severity
severity_mapping = {
    'minor' : 0,
    'major' : 1,
    'fatal' : 2
}
# A dictionary maps each text value to a number
# minor=0, major=1, fatal=2 — this ordering is intentional
# it reflects real-world severity: minor < major < fatal

pdf_clean['severity_encoded'] = pdf_clean['accident_severity'].map(severity_mapping)
# .map(dictionary) → replaces each value using the dictionary
# "minor" → 0, "major" → 1, "fatal" → 2

# Keep the original column for reference — we'll drop it before modelling
# Verify
print("Severity encoding check:")
print(pdf_clean[['accident_severity', 'severity_encoded']].value_counts())

In [0]:
# These are text columns the model needs as numbers
cat_columns = [
    'city', 'road_type', 'weather',
    'visibility', 'traffic_density', 'cause', 'day_of_week'
]

# We use Label Encoding — assigns a number to each unique text value
# e.g. clear=0, fog=1, rain=2
from sklearn.preprocessing import LabelEncoder
# LabelEncoder is a tool from scikit-learn
# it automatically assigns a number to each unique category

le = LabelEncoder()
# le is our encoder object — think of it as a translator

for col in cat_columns:
    pdf_clean[col + '_encoded'] = le.fit_transform(pdf_clean[col])
    # fit_transform does two things:
    #   fit          → learns all unique values in the column
    #   transform    → converts each value to its assigned number
    # col + '_encoded' → creates a new column e.g. 'weather_encoded'
    #                    keeping original column intact for reference
    
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")
    # le.classes_          → the unique values it learned
    # le.transform(...)    → their corresponding numbers
    # zip(...) + dict(...) → combines them into a readable dictionary
    # so you can see exactly what number maps to what value

In [0]:
# See what our clean dataset looks like now
print(f"Shape: {pdf_clean.shape}")
print(f"\nColumn list:")
for col in pdf_clean.columns:
    print(f"  {col:<35} {str(pdf_clean[col].dtype):<15}")
    
print(f"\nMissing values: {pdf_clean.isnull().sum().sum()}")
# .sum().sum() → first sums nulls per column, then sums those totals
# gives one single number — total nulls in entire dataset

In [0]:
# Fix visibility — natural order: low=0, medium=1, high=2
visibility_mapping = {'low': 0, 'medium': 1, 'high': 2}
pdf_clean['visibility_encoded'] = pdf_clean['visibility'].map(visibility_mapping)
# We override the alphabetical encoding with a meaningful one
# low visibility is most dangerous → 0
# high visibility is safest        → 2
# .map() replaces each text value using our custom dictionary

# Fix traffic_density — natural order: low=0, medium=1, high=2
traffic_mapping = {'low': 0, 'medium': 1, 'high': 2}
pdf_clean['traffic_density_encoded'] = pdf_clean['traffic_density'].map(traffic_mapping)

# Fix day_of_week — natural order: Monday=0 ... Sunday=6
day_mapping = {
    'Monday'   : 0,
    'Tuesday'  : 1,
    'Wednesday': 2,
    'Thursday' : 3,
    'Friday'   : 4,
    'Saturday' : 5,
    'Sunday'   : 6
}
pdf_clean['day_of_week_encoded'] = pdf_clean['day_of_week'].map(day_mapping)

# Verify all three
print("Visibility encoding:")
print(pdf_clean[['visibility', 'visibility_encoded']].drop_duplicates().sort_values('visibility_encoded'))
print("\nTraffic density encoding:")
print(pdf_clean[['traffic_density', 'traffic_density_encoded']].drop_duplicates().sort_values('traffic_density_encoded'))
print("\nDay of week encoding:")
print(pdf_clean[['day_of_week', 'day_of_week_encoded']].drop_duplicates().sort_values('day_of_week_encoded'))
# drop_duplicates() → removes repeated rows so we see one row per unique value
# sort_values()     → sorts by the encoded number so it's easy to read

In [0]:
# ── Feature 1: Danger Score ──────────────────────────────────────────
# Combines visibility, traffic density and weather into one risk number
pdf_clean['danger_score'] = (
    pdf_clean['visibility_encoded'] * -1 + 2  +  # low visibility = high danger
    pdf_clean['traffic_density_encoded']       +  # high traffic = high danger
    pdf_clean['weather_encoded']                  # fog/rain = high danger
)
# We flip visibility because low=0 should mean high danger
# visibility_encoded * -1 + 2 → low(0)→2, medium(1)→1, high(2)→0
# Then we add traffic and weather scores on top
# Higher danger_score = more dangerous conditions overall

# ── Feature 2: Night time flag ───────────────────────────────────────
pdf_clean['is_night'] = ((pdf_clean['hour'] >= 20) | (pdf_clean['hour'] <= 5)).astype(int)
# hour >= 20 means 8pm or later
# hour <= 5  means 5am or earlier
# | means OR — either condition makes it nighttime
# .astype(int) converts True/False to 1/0

# ── Feature 3: High risk cause flag ─────────────────────────────────
high_risk_causes = ['drunk driving', 'overspeeding']
pdf_clean['is_high_risk_cause'] = pdf_clean['cause'].isin(high_risk_causes).astype(int)
# .isin([...]) checks if the value is in our list
# returns True for drunk driving and overspeeding, False for others
# these are objectively more dangerous causes

# ── Feature 4: Busy road flag ────────────────────────────────────────
pdf_clean['is_busy_road'] = (
    (pdf_clean['lanes'] >= 3) & 
    (pdf_clean['traffic_density_encoded'] >= 1)
).astype(int)
# & means AND — both conditions must be true
# 3+ lanes AND at least medium traffic = busy road

# ── Feature 5: Bad conditions flag ──────────────────────────────────
pdf_clean['is_bad_conditions'] = (
    (pdf_clean['weather'] != 'clear') & 
    (pdf_clean['visibility'] == 'low')
).astype(int)
# Not clear weather AND low visibility = genuinely bad conditions
# Both must be true at the same time

# Verify new features
print("New features created:")
print(pdf_clean[['danger_score', 'is_night', 'is_high_risk_cause', 
                  'is_busy_road', 'is_bad_conditions']].describe())
# .describe() gives count, mean, min, max etc. for each column
# quick sanity check that values are in expected ranges

In [0]:
# Columns to drop — text originals we no longer need
cols_to_drop = [
    'city', 'road_type', 'weather', 'visibility',
    'traffic_density', 'cause', 'day_of_week',
    'accident_severity'   # we have severity_encoded instead
]

pdf_model = pdf_clean.drop(columns=cols_to_drop)
# pdf_model is our final clean dataset ready for the model
# only numbers, no text

print(f"Final model dataset shape: {pdf_model.shape}")
print(f"\nFinal columns:")
for col in pdf_model.columns:
    print(f"  {col}")

In [0]:
from pyspark.sql import SparkSession

# Convert Pandas back to Spark DataFrame
spark_df = spark.createDataFrame(pdf_model)
# spark.createDataFrame() → converts our Pandas df back to Spark
# we need Spark format to write to Unity Catalog

# Write to Unity Catalog processed schema
spark_df.write \
    .mode("overwrite") \
    .saveAsTable("indian_roads.processed.accidents_clean")
# .mode("overwrite") → replaces the table if it already exists
# .saveAsTable()     → saves as a proper Delta table in Unity Catalog
# now anyone on the team can access the clean data from here

print("✅ Clean data saved to indian_roads.processed.accidents_clean")

# Verify it saved correctly
verify = spark.table("indian_roads.processed.accidents_clean")
print(f"Rows in saved table: {verify.count()}")
print(f"Columns in saved table: {len(verify.columns)}")

In [0]:
# Cell 1 — Imports (always goes at the top of every notebook)
import pandas as pd
import numpy as np

# Load raw data from Unity Catalog
df = spark.table("indian_roads.raw.accidents")

# Convert to Pandas
pdf = df.toPandas()

print(f"Shape: {pdf.shape}")

In [0]:
# Convert date from string to proper datetime
pdf['date'] = pd.to_datetime(pdf['date'])
# pd.to_datetime() → tells pandas "treat this column as a real date"
# Before: "2023-10-22" was just text
# After:  it becomes a proper date object we can do maths on

# Extract useful date features
pdf['month']      = pdf['date'].dt.month
pdf['year']       = pdf['date'].dt.year
pdf['day']        = pdf['date'].dt.day

# .dt is the "datetime accessor" — lets you pull parts out of a date
# .dt.month → extracts just the month number (1–12)
# .dt.year  → extracts just the year
# .dt.day   → extracts just the day of the month

print("Date column fixed:")
print(pdf[['date', 'year', 'month', 'day']].head(5))

In [0]:
cols_to_drop = [
    'accident_id',   # just a row number, no predictive value
    'time',          # hour column already captures this
    'date',          # we extracted year/month/day, raw date not needed
    'latitude',      # too granular, city already captures location
    'longitude',     # same reason
    'state',         # city is more specific, state is redundant
    'risk_score'     # this was likely calculated FROM severity — using it would be cheating
]

pdf_clean = pdf.drop(columns=cols_to_drop)
# .drop(columns=[...]) removes the listed columns
# we save to pdf_clean to keep the original pdf untouched
# good practice — never overwrite raw data

print(f"Columns before: {pdf.shape[1]}")
print(f"Columns after:  {pdf_clean.shape[1]}")
print(f"\nRemaining columns:\n{list(pdf_clean.columns)}")